# パッケージをインストール

In [1]:
!pip install -qU \
    langchain-qdrant \
    qdrant-client \
    fastembed \
    sudachipy \
    sudachidict_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 29.7 MB/s eta 0:00:00


In [2]:
import importlib.metadata

packages = [
    "langchain-qdrant",
    "qdrant-client",
    "fastembed",
    "SudachiPy",
    "SudachiDict-core",
]

for package in packages:
    try:
        print(f"{package}: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: not found")

langchain-qdrant: 1.1.0
qdrant-client: 1.19.0
fastembed: 0.8.0
SudachiPy: 0.6.11
SudachiDict-core: 20260723


# テスト用Documentを作成

In [3]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=(
            "Pythonを3年間使用し、AWS Lambdaを利用した"
            "サーバーレスアプリケーションの設計・開発経験があります。"
        ),
        metadata={"id": "engineer-001"},
    ),
    Document(
        page_content=(
            "JavaとSpring Bootを用いたWeb API開発を5年間経験しています。"
            "MySQLやDockerも使用できます。"
        ),
        metadata={"id": "engineer-002"},
    ),
    Document(
        page_content=(
            "C++による画像処理システムの開発経験があります。"
            "CUDA、TensorRT、Jetson Orinを利用したエッジAI開発を担当しました。"
        ),
        metadata={"id": "engineer-003"},
    ),
    Document(
        page_content=(
            "TypeScript、React、Next.jsを用いたフロントエンド開発経験があります。"
            "AWS環境でのシステム開発も経験しています。"
        ),
        metadata={"id": "engineer-004"},
    ),
    Document(
        page_content=(
            "Pythonを使用してRAGシステムを開発しました。"
            "LangChain、Qdrant、ローカルLLMを利用した検索システムの"
            "設計と精度評価を経験しています。"
        ),
        metadata={"id": "engineer-005"},
    ),
    Document(
        page_content=(
            "COBOLを使用した銀行勘定系システムの開発と保守を"
            "10年間担当しました。"
        ),
        metadata={"id": "engineer-006"},
    ),
]

print(f"documents: {len(documents)}")

documents: 6


# SudachiPy tokenizerを作成

In [4]:
from sudachipy import Dictionary, SplitMode


class SudachiBM25Tokenizer:
    """
    BM25検索用のSudachi tokenizer。

    - SplitMode.C
    - 名詞を抽出
    - normalized_form() を使用
    """

    def __init__(
        self,
        split_mode=SplitMode.C,
        target_pos: set[str] | None = None,
    ):
        self.tokenizer = Dictionary().create()
        self.split_mode = split_mode

        # BM25のキーワード検索では、まず名詞中心から開始
        self.target_pos = target_pos or {
            "名詞",
        }

    def __call__(self, text: str) -> list[str]:
        tokens = []

        for morpheme in self.tokenizer.tokenize(
            text,
            self.split_mode,
        ):
            pos = morpheme.part_of_speech()[0]

            if pos not in self.target_pos:
                continue

            token = morpheme.normalized_form().strip()

            if not token:
                continue

            tokens.append(token)

        return tokens


tokenize = SudachiBM25Tokenizer()

In [5]:
test_text = (
    "PythonとC++を使用し、AWS LambdaとTensorRTを利用した"
    "AIシステムの開発経験があります。"
)

tokens = tokenize(test_text)

print("Original:")
print(test_text)

print("\nSudachi tokens:")
print(tokens)

Original:
PythonとC++を使用し、AWS LambdaとTensorRTを利用したAIシステムの開発経験があります。

Sudachi tokens:
['Python', 'c', '使用', 'AWS', 'lambda', 'tensorrt', '利用', 'AI', 'システム', '開発', '経験']


# 全DocumentのSudachi結果を確認

In [6]:
for doc in documents:
    tokens = tokenize(doc.page_content)

    print("=" * 80)
    print("ID:", doc.metadata["id"])
    print("Original:")
    print(doc.page_content)

    print("\nTokens:")
    print(tokens)

ID: engineer-001
Original:
Pythonを3年間使用し、AWS Lambdaを利用したサーバーレスアプリケーションの設計・開発経験があります。

Tokens:
['python', '3', '年間', '使用', 'AWS', 'lambda', '利用', 'サーバーレスアプリケーション', '設計', '開発', '経験']
ID: engineer-002
Original:
JavaとSpring Bootを用いたWeb API開発を5年間経験しています。MySQLやDockerも使用できます。

Tokens:
['Java', 'スプリング', 'ブート', 'ウェブ', 'API', '開発', '5', '年間', '経験', 'mysql', 'Docker', '使用']
ID: engineer-003
Original:
C++による画像処理システムの開発経験があります。CUDA、TensorRT、Jetson Orinを利用したエッジAI開発を担当しました。

Tokens:
['c', '画像', '処理', 'システム', '開発', '経験', 'cuda', 'tensorrt', 'jetson', 'orin', '利用', 'エッジAI', '開発', '担当']
ID: engineer-004
Original:
TypeScript、React、Next.jsを用いたフロントエンド開発経験があります。AWS環境でのシステム開発も経験しています。

Tokens:
['typescript', 'react', 'ネクスト', 'js', 'フロントエンド', '開発', '経験', 'AWS', '環境', 'システム', '開発', '経験']
ID: engineer-005
Original:
Pythonを使用してRAGシステムを開発しました。LangChain、Qdrant、ローカルLLMを利用した検索システムの設計と精度評価を経験しています。

Tokens:
['python', '使用', 'rag', 'システム', '開発', 'langchain', 'qdrant', 'ローカル', 'LLM', '利用', '検索', 'システム', '設計', '精度', '評価

# avg_lenを計算
FastEmbed のQdrant BM25では avg_len のデフォルトは256ですが、実データから計算した方が自然です。Qdrant公式も、対象フィールドの平均トークン長に応じて avg_len を調整することを推奨しています。

In [7]:
tokenized_documents = [
    tokenize(doc.page_content)
    for doc in documents
]

document_lengths = [
    len(tokens)
    for tokens in tokenized_documents
]

avg_len = (
    sum(document_lengths)
    / len(document_lengths)
)

print("Document lengths:")
print(document_lengths)

print()
print(f"avg_len = {avg_len:.3f}")

Document lengths:
[11, 12, 14, 12, 16, 10]

avg_len = 12.500


# JapaneseBM25SparseEmbeddings を実装

In [8]:
from __future__ import annotations

from collections.abc import Callable

from langchain_qdrant import (
    FastEmbedSparse,
    SparseEmbeddings,
    SparseVector,
)


class JapaneseBM25SparseEmbeddings(SparseEmbeddings):
    """
    SudachiPy + FastEmbedSparse(Qdrant/bm25)

    処理:

        日本語
          ↓
        SudachiPy
          ↓
        token list
          ↓
        FastEmbedで安全に扱える文字列へ変換
          ↓
        FastEmbedSparse
          ↓
        SparseVector
    """

    def __init__(
        self,
        tokenize: Callable[[str], list[str]],
        avg_len: float,
        *,
        k: float = 1.5,
        b: float = 0.75,
    ):
        self.tokenize = tokenize
        self.avg_len = avg_len
        self.k = k
        self.b = b

        self.bm25 = FastEmbedSparse(
            model_name="Qdrant/bm25",

            # 日本語用stemmerは使用しない
            disable_stemmer=True,

            # rank_bm25.BM25Okapiに近い値から開始
            k=k,
            b=b,

            # 実コーパスから計算
            avg_len=avg_len,

            # tokenをhex化するので長めにする
            token_max_length=512,
        )

    @staticmethod
    def _encode_token(token: str) -> str:
        """
        FastEmbed内部の記号除去からtokenを保護する。

        C++
        ↓
        UTF-8
        ↓
        t432b2b

        のような英数字だけのtokenへ変換する。

        DocumentとQueryの両方に同じ処理を行うので
        BM25検索上は同じtokenとして扱える。
        """

        return "t" + token.encode("utf-8").hex()

    def tokenize_text(
        self,
        text: str,
    ) -> list[str]:
        """
        デバッグ用。
        Sudachiのtokenをそのまま返す。
        """

        return self.tokenize(text)

    def preprocess(
        self,
        text: str,
    ) -> str:
        """
        Sudachi token
        ↓
        FastEmbed入力文字列
        """

        tokens = self.tokenize(text)

        safe_tokens = [
            self._encode_token(token)
            for token in tokens
        ]

        return " ".join(safe_tokens)

    def embed_documents(
        self,
        texts: list[str],
    ) -> list[SparseVector]:

        processed = [
            self.preprocess(text)
            for text in texts
        ]

        return self.bm25.embed_documents(
            processed
        )

    def embed_query(
        self,
        text: str,
    ) -> SparseVector:

        processed = self.preprocess(text)

        return self.bm25.embed_query(
            processed
        )

In [10]:
sparse_embeddings = JapaneseBM25SparseEmbeddings(
    tokenize=tokenize,
    avg_len=avg_len,
    k=1.5,
    b=0.75,
)

# 前処理結果を確認

In [11]:
test_text = (
    "C++とTensorRTを使用した"
    "Jetsonでの画像AI開発経験"
)

print("Original:")
print(test_text)

print("\nSudachi:")
print(
    sparse_embeddings.tokenize_text(
        test_text
    )
)

print("\nFastEmbed input:")
print(
    sparse_embeddings.preprocess(
        test_text
    )
)

Original:
C++とTensorRTを使用したJetsonでの画像AI開発経験

Sudachi:
['c', 'tensorrt', '使用', 'jetson', '画像', 'AI', '開発', '経験']

FastEmbed input:
t63 t74656e736f727274 te4bdbfe794a8 t6a6574736f6e te794bbe5838f t4149 te9968be799ba te7b58ce9a893


# SparseVectorそのものを確認

In [12]:
text = documents[0].page_content

vector = sparse_embeddings.embed_documents(
    [text]
)[0]

print("Document:")
print(text)

print("\nSudachi:")
print(
    sparse_embeddings.tokenize_text(text)
)

print("\nSparse vector:")
print("indices:")
print(vector.indices)

print("\nvalues:")
print(vector.values)

print()
print(
    "non-zero dimensions:",
    len(vector.indices),
)

Document:
Pythonを3年間使用し、AWS Lambdaを利用したサーバーレスアプリケーションの設計・開発経験があります。

Sudachi:
['python', '3', '年間', '使用', 'AWS', 'lambda', '利用', 'サーバーレスアプリケーション', '設計', '開発', '経験']

Sparse vector:
indices:
[566669935, 1243156251, 1663240087, 1039520159, 250604404, 2122659614, 263643576, 1298292464, 606807848, 118033296, 876722907]

values:
[1.0570824524312896, 1.0570824524312896, 1.0570824524312896, 1.0570824524312896, 1.0570824524312896, 1.0570824524312896, 1.0570824524312896, 1.0570824524312896, 1.0570824524312896, 1.0570824524312896, 1.0570824524312896]

non-zero dimensions: 11


# Query側も確認

In [13]:
query = "Python AWS Lambda 開発経験"

query_vector = sparse_embeddings.embed_query(
    query
)

print("Query:")
print(query)

print("\nSudachi:")
print(
    sparse_embeddings.tokenize_text(query)
)

print("\nFastEmbed input:")
print(
    sparse_embeddings.preprocess(query)
)

print("\nQuery SparseVector:")
print("indices:")
print(query_vector.indices)

print("\nvalues:")
print(query_vector.values)

Query:
Python AWS Lambda 開発経験

Sudachi:
['Python', 'AWS', 'lambda', '開発', '経験']

FastEmbed input:
t507974686f6e t415753 t6c616d626461 te9968be799ba te7b58ce9a893

Query SparseVector:
indices:
[1631749188, 118033296, 250604404, 876722907, 2122659614]

values:
[1.0, 1.0, 1.0, 1.0, 1.0]


# DocumentとQueryで同じtoken IDになることを確認

In [14]:
document_text = "Python AWS Lambdaを利用した開発経験"
query_text = "Python AWS"

document_vector = sparse_embeddings.embed_documents(
    [document_text]
)[0]

query_vector = sparse_embeddings.embed_query(
    query_text
)

document_indices = set(document_vector.indices)
query_indices = set(query_vector.indices)

common_indices = (
    document_indices
    & query_indices
)

print("Document tokens:")
print(
    sparse_embeddings.tokenize_text(
        document_text
    )
)

print("\nQuery tokens:")
print(
    sparse_embeddings.tokenize_text(
        query_text
    )
)

print("\nDocument indices:")
print(document_vector.indices)

print("\nQuery indices:")
print(query_vector.indices)

print("\nCommon indices:")
print(common_indices)

print(
    "\nCommon token count:",
    len(common_indices),
)

Document tokens:
['Python', 'AWS', 'lambda', '利用', '開発', '経験']

Query tokens:
['Python', 'AWS']

Document indices:
[1631749188, 250604404, 2122659614, 263643576, 118033296, 876722907]

Query indices:
[250604404, 1631749188]

Common indices:
{250604404, 1631749188}

Common token count: 2


# Qdrantをインメモリ起動

In [15]:
from qdrant_client import (
    QdrantClient,
    models,
)

client = QdrantClient(":memory:")

COLLECTION_NAME = "engineer_bm25_demo"
SPARSE_VECTOR_NAME = "bm25"

# Sparse Collectionを作成

ここで非常に重要なのが Modifier.IDF です。

FastEmbedの Qdrant/bm25 はBM25のTF部分をSparseVectorへ入れ、IDFはQdrant側で計算する設計です。そのため、Sparse Vector設定に modifier=IDF が必要です。FastEmbedの実装にも明記されています。
https://github.com/qdrant/fastembed/blob/main/fastembed/sparse/bm25.py

In [16]:
client.create_collection(
    collection_name=COLLECTION_NAME,

    sparse_vectors_config={
        SPARSE_VECTOR_NAME:
            models.SparseVectorParams(
                modifier=models.Modifier.IDF,
            )
    },
)

print(
    client.get_collection(
        COLLECTION_NAME
    )
)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=0 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors={}, shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, payload=None, sparse_vectors={'bm25': SparseVectorParams(index=None, modifier=<Modifier.IDF: 'idf'>)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1, prevent_unoptimized=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0, wal_retain_cl

# QdrantVectorStore と接続

In [17]:
from langchain_qdrant import (
    QdrantVectorStore,
    RetrievalMode,
)


vector_store = QdrantVectorStore(
    client=client,

    collection_name=COLLECTION_NAME,

    sparse_embedding=sparse_embeddings,

    sparse_vector_name=SPARSE_VECTOR_NAME,

    retrieval_mode=RetrievalMode.SPARSE,
)

# DocumentをQdrantへ登録

In [18]:
ids = vector_store.add_documents(
    documents=documents
)

print("Inserted IDs:")

for id_ in ids:
    print(id_)

Inserted IDs:
6483a5299a434fc89fd35e479b8c5f72
9af7bac8d32e4f8395b7b0544a3eef69
488c91caa566483e9a90b1618c4fb09f
0deed57b0d644df59289fec1e99bfe48
79ade3537bfa42bcab774387283dea56
f9ed07823d214d4d8da17b70c9dbcaeb


# BM25 Sparse検索

In [19]:
query = "Python AWS Lambdaを利用した開発経験"

results = vector_store.similarity_search_with_score(
    query=query,
    k=3,
)

print("Query:")
print(query)

print("\nSearch results:")

for rank, (doc, score) in enumerate(
    results,
    start=1,
):
    print("=" * 80)

    print(
        f"Rank: {rank}"
    )

    print(
        f"ID: {doc.metadata['id']}"
    )

    print(
        f"Score: {score:.6f}"
    )

    print(
        f"Text: {doc.page_content}"
    )

Query:
Python AWS Lambdaを利用した開発経験

Search results:
Rank: 1
ID: engineer-001
Score: 3.782750
Text: Pythonを3年間使用し、AWS Lambdaを利用したサーバーレスアプリケーションの設計・開発経験があります。
Rank: 2
ID: engineer-004
Score: 1.504744
Text: TypeScript、React、Next.jsを用いたフロントエンド開発経験があります。AWS環境でのシステム開発も経験しています。
Rank: 3
ID: engineer-003
Score: 0.988378
Text: C++による画像処理システムの開発経験があります。CUDA、TensorRT、Jetson Orinを利用したエッジAI開発を担当しました。


# C++ / TensorRT検索

In [20]:
query = (
    "C++ TensorRT Jetson "
    "画像AI開発"
)

results = vector_store.similarity_search_with_score(
    query=query,
    k=3,
)

print("Query:")
print(query)

print()

for rank, (doc, score) in enumerate(
    results,
    start=1,
):
    print(
        f"{rank}. "
        f"{doc.metadata['id']} "
        f"score={score:.6f}"
    )

    print(
        doc.page_content
    )

    print()

Query:
C++ TensorRT Jetson 画像AI開発

1. engineer-003 score=5.948028
C++による画像処理システムの開発経験があります。CUDA、TensorRT、Jetson Orinを利用したエッジAI開発を担当しました。

2. engineer-004 score=0.107247
TypeScript、React、Next.jsを用いたフロントエンド開発経験があります。AWS環境でのシステム開発も経験しています。

3. engineer-006 score=0.081437
COBOLを使用した銀行勘定系システムの開発と保守を10年間担当しました。



# RAG / Qdrant検索

In [21]:
query = (
    "Python Qdrant LangChain "
    "RAGシステム"
)

results = vector_store.similarity_search_with_score(
    query=query,
    k=3,
)

for rank, (doc, score) in enumerate(
    results,
    start=1,
):
    print(
        f"{rank}. "
        f"{doc.metadata['id']} "
        f"score={score:.6f}"
    )

    print(
        doc.page_content
    )

    print()

1. engineer-005 score=4.683279
Pythonを使用してRAGシステムを開発しました。LangChain、Qdrant、ローカルLLMを利用した検索システムの設計と精度評価を経験しています。

2. engineer-006 score=0.485530
COBOLを使用した銀行勘定系システムの開発と保守を10年間担当しました。

3. engineer-004 score=0.449932
TypeScript、React、Next.jsを用いたフロントエンド開発経験があります。AWS環境でのシステム開発も経験しています。



# Retriever として使う

In [22]:
retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 3,
    }
)
query = "AWS Lambda Python"

docs = retriever.invoke(query)

for rank, doc in enumerate(
    docs,
    start=1,
):
    print(
        f"{rank}. {doc.metadata['id']}"
    )

    print(doc.page_content)

    print()

1. engineer-001
Pythonを3年間使用し、AWS Lambdaを利用したサーバーレスアプリケーションの設計・開発経験があります。

2. engineer-004
TypeScript、React、Next.jsを用いたフロントエンド開発経験があります。AWS環境でのシステム開発も経験しています。

